In [138]:
import torch
from torch import nn


In [139]:
!nvidia-smi

Sat Jul 25 12:05:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 566.07                 Driver Version: 566.07         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3060 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   54C    P8             17W /  140W |    1851MiB /   6144MiB |     31%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [140]:
import torch
from torch import nn

torch.device('cpu'), torch.device('cuda'), torch.device('cuda:1')

(device(type='cpu'), device(type='cuda'), device(type='cuda', index=1))

In [141]:
torch.cuda.device_count()

1

In [142]:
def try_gpu(i=0):
    if torch.cuda.device_count()>= i+1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')
try_gpu(2)

device(type='cpu')

In [143]:
def try_all_gpu():
    device=[torch.device(f'cuda:{i}')
     for i in range(torch.cuda.device_count())
     ]
    return device if device else [torch.device('cpu')]
try_all_gpu()

[device(type='cuda', index=0)]

In [144]:
x=torch.arange(12.0,requires_grad=True)
y=torch.arange(12.0,device=try_gpu(),requires_grad=True)
z=x.to(try_gpu())
u=y+z
print(u)
u.sum().backward()
print(x.grad)
print(y.grad)
print(z.grad)

tensor([ 0.,  2.,  4.,  6.,  8., 10., 12., 14., 16., 18., 20., 22.],
       device='cuda:0', grad_fn=<AddBackward0>)
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], device='cuda:0')
None


In [145]:
net=nn.Sequential(nn.Linear(20,20))
o=torch.randn(size=(20,20),device=try_gpu())
net=net.to(try_gpu())
print(net(o))


tensor([[ 1.1097e+00, -2.0132e-01,  1.3251e-02,  1.4405e-01, -1.0831e+00,
          1.9800e-01, -2.7949e-01,  1.1581e+00, -4.2064e-01,  3.5701e-01,
         -3.8446e-01, -6.8147e-01,  4.8738e-02,  1.0660e+00, -2.3521e-01,
         -1.9772e-01, -9.9025e-01,  4.3501e-01, -8.7089e-01,  1.3095e+00],
        [-9.0156e-02,  1.3150e-01,  1.1525e+00, -9.1421e-01,  1.6621e-01,
          5.8994e-01, -1.4245e-02, -2.1779e-01,  4.3666e-01,  4.8445e-01,
          7.9787e-01,  4.1231e-01,  3.2752e-01, -6.3157e-02,  1.1749e+00,
         -1.8612e-01, -3.0940e-01,  4.1027e-01,  2.7650e-01,  3.4846e-01],
        [ 5.4953e-02, -6.2601e-01,  1.8945e-01,  9.0125e-01, -3.1116e-01,
          4.4050e-01, -2.6018e-01,  5.3053e-01,  4.3301e-01,  8.5814e-01,
         -9.0417e-01,  4.5856e-01, -6.3304e-01,  6.2751e-01, -6.3015e-01,
         -4.7432e-01, -2.3186e-01,  1.3595e+00, -5.2769e-01,  3.2113e-01],
        [ 1.6102e-01,  2.5430e-01,  5.4197e-02,  5.3153e-01,  4.5907e-01,
          4.5687e-01,  7.7926e-03, 

In [146]:
net[0].weight.data.device

device(type='cuda', index=0)

In [ ]:
class mynet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net11=nn.Sequential()
        self.net11.add_module("1",nn.Linear(20,20))
    def forward(self,x):
        for layer in self.children():
            x=layer(x)
        return x
net1=mynet()
net1=net1.to(try_gpu())
state=torch.load('net_data') #之前这个文件是在cpu上保存 所以恢复到cpu上
net1.load_state_dict(state)

<All keys matched successfully>

In [150]:
print(next(iter(state.values())).device)
print(next(net1.parameters()).device)

cpu
cuda:0
